# Accumulator Variables

- Accumulators are special variables for aggregating data across nodes.
- They are mainly used for counting or summing values in parallel tasks.
- Each task has its own accumulator variable. The tasks can only write to an accumulator variable.
- Driver node is responsible to aggregate the accumulator variables from each task and provide the final result.
- If a task runs again due to some issue, accumulator updates may be duplicated. 🚨
- Use accumulators for debugging or monitoring, not core logic.
- Spark supports numeric accumulators and allows custom types.


## Syntax
```python
# Define the accumulator variable
acc = spark.sparkContext.accumulator(0)

# Read the accumulator value
acc_value = acc.value
```

In [6]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()

In [9]:
log_rdd = spark.sparkContext.textFile('/content/application.log')
log_rdd.take(5)

['2012-02-03 18:35:34 SampleClass6 [INFO] everything normal for id 577725851',
 '2012-02-03 18:35:34 SampleClass4 [FATAL] system problem at id 1991281254',
 '2012-02-03 18:35:34 SampleClass3 [DEBUG] detail for id 1304807656',
 '2012-02-03 18:35:34 SampleClass3 [WARN] missing id 423340895',
 '2012-02-03 18:35:34 SampleClass5 [TRACE] verbose detail for id 2082654978']

In [39]:
log_rdd.count()

1387

## Problem statement
- Get count of each LogLevel
- Count of invalid lines
- Count of empty lines

### Without using accumulator

In [41]:
def getLogLevel(log_iter):
  valid_logLevels = ['[INFO]', '[FATAL]', '[DEBUG]', '[WARN]', '[TRACE]', '[ERROR]']
  for log_line in log_iter:
    log_line = log_line.strip()
    if log_line:
      log_line_list = log_line.split(' ')
      if len(log_line_list) >= 4:
        log_level = log_line_list[3]
        if log_level.upper() in valid_logLevels:
          # For valid logLevels yield will be executed
          pass
        else:
          # For invalid logLevels skip yield
          continue
      else:
        # For invalid lines (Length < 4) skip yield
        continue
    else:
      # For empty lines skip yield
      continue
    yield log_level



def getInvalidLines(log_iter):
  valid_logLevels = ['[INFO]', '[FATAL]', '[DEBUG]', '[WARN]', '[TRACE]', '[ERROR]']
  for log_line in log_iter:
    log_line = log_line.strip()
    if log_line:
      log_line_list = log_line.split(' ')
      if len(log_line_list) >= 4:
        log_level = log_line_list[3]
        if log_level.upper() not in valid_logLevels:
          # For invalid logLevels yield will be executed
          pass
        else:
          # For valid logLevels skip yield
          continue
      else:
        # For invalid lines (Length < 4) yield will be executed
        pass
    else:
      # For empty lines skip yield
      continue
    yield 1



def getEmptyLines(log_iter):
  for log_line in log_iter:
    log_line = log_line.strip()
    if log_line:
      # For non empty lines skip yield
      continue
    else:
      # For empty lines yield will be executed
      pass
    yield 1

In [42]:
logLevel_count = log_rdd.mapPartitions(getLogLevel).countByValue()
logLevel_count

defaultdict(int,
            {'[INFO]': 96,
             '[FATAL]': 1,
             '[DEBUG]': 434,
             '[WARN]': 4,
             '[TRACE]': 816,
             '[ERROR]': 3})

In [43]:
invalid_count = log_rdd.mapPartitions(getInvalidLines).count()
invalid_count

22

In [44]:
empty_count = log_rdd.mapPartitions(getEmptyLines).count()
empty_count

11

### Using accumulator

In [47]:
def getAllStats(log_iter):
  valid_logLevels = ['[INFO]', '[FATAL]', '[DEBUG]', '[WARN]', '[TRACE]', '[ERROR]']
  global invalid_count_acc, empty_count_acc
  for log_line in log_iter:
    log_line = log_line.strip()
    if log_line:
      log_line_list = log_line.split(' ')
      if len(log_line_list) >= 4:
        log_level = log_line_list[3]
        if log_level.upper() in valid_logLevels:
          # For valid logLevels yield log_level
          yield log_level
        else:
          # For invalid logLevels increment invalid_count_acc
          invalid_count_acc += 1
      else:
        # For invalid lines (Length < 4) increment invalid_count_acc
        invalid_count_acc += 1
    else:
      # For empty lines increment empty_count_acc
      empty_count_acc += 1

In [48]:
# Define the accumulator variables
invalid_count_acc = spark.sparkContext.accumulator(0)
empty_count_acc = spark.sparkContext.accumulator(0)

logLevel_count = log_rdd.mapPartitions(getAllStats).countByValue()
print(f'{logLevel_count} \n Invalid Count = {invalid_count_acc.value} \n Empty Count = {empty_count_acc.value}')

defaultdict(<class 'int'>, {'[INFO]': 96, '[FATAL]': 1, '[DEBUG]': 434, '[WARN]': 4, '[TRACE]': 816, '[ERROR]': 3}) 
 Invalid Count = 22 
 Empty Count = 11
